# Калибровка ядра `ddeco`

Notebook — слой **поверх** пакета `ddeco`: логика симуляции живёт в `ddeco/`, здесь только
прогоны, графики и подбор параметров. Если меняешь модель — правь `ddeco/`, не ячейки.

Сейчас проверяем одноузловой контур цена–запас–производство (MODEL.md §6) и сверяем скорость
сходимости со спектральными радиусами из таблицы §6.3.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # запуск из notebooks/
import numpy as np
import matplotlib.pyplot as plt

from ddeco.tick import step_day
from tests.single_node import single_node_world, perturbed_state, NODE, GOOD

In [ ]:
def trajectory(alpha, beta, eps, T, ratio=0.8, lam=0.3, days=400):
    world = single_node_world(alpha, beta, eps, T, lam=lam)
    state, star = perturbed_state(world, ratio)
    s, m = [], []
    for _ in range(days):
        step_day(world, state)
        s.append(state.stock[(NODE, GOOD)] / star)
        m.append(state.multiplier[(NODE, GOOD)])
    return np.array(s), np.array(m)

# (имя, α, β, ε, T, предсказанный ρ из §6.3)
CATS = [
    ('necessity', 0.5, 0.3, 0.2, 10, 0.973),
    ('common',    1.0, 0.3, 0.6, 10, 0.837),
    ('luxury',    1.5, 0.3, 1.5, 14, 0.837),
]

## Сходимость запаса к целевому (s/s* → 1) после возмущения −20%

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for name, a, b, e, T, _ in CATS:
    s, m = trajectory(a, b, e, T)
    ax1.plot(s, label=name)
    ax2.plot(m, label=name)
ax1.axhline(1.0, color='k', lw=0.5, ls='--'); ax1.set_title('s / s*'); ax1.set_xlabel('день'); ax1.legend()
ax2.axhline(1.0, color='k', lw=0.5, ls='--'); ax2.set_title('множитель цены m'); ax2.set_xlabel('день'); ax2.legend()
plt.tight_layout(); plt.show()

## Сверка скорости сходимости со спектральным радиусом §6.3

Отклонение |s/s* − 1| должно падать как ρ^t. Строим в лог-шкале; наклон = ln ρ.
Для `necessity` измеренный ρ ≈ 0.973 совпадает с таблицей; `common`/`luxury` затухают
быстрее (колебательно) и достигают машинного нуля за ~200 дней.

In [ ]:
plt.figure(figsize=(8, 5))
for name, a, b, e, T, rho in CATS:
    s, _ = trajectory(a, b, e, T, days=300)
    dev = np.abs(s - 1.0)
    plt.semilogy(dev, label=f'{name} (ρ_пред={rho})')
    plt.semilogy(dev[0] * rho ** np.arange(len(dev)), ls=':', color='gray')
plt.title('|s/s* − 1| (лог) и предсказанный ρ^t (пунктир)')
plt.xlabel('день'); plt.ylabel('отклонение'); plt.ylim(1e-6, 1); plt.legend()
plt.tight_layout(); plt.show()

## Песочница: влияние λ на колебания

Стресс-категория (α=2, β=0.5, ε=1.5, T=2 → a=2) на границе устойчивости: при λ=1 —
незатухающие колебания, при λ=0.5 — затухают (§6.3).

In [ ]:
plt.figure(figsize=(8, 4))
for lam in (1.0, 0.5, 0.3):
    s, _ = trajectory(2.0, 0.5, 1.5, 2.0, lam=lam, days=120)
    plt.plot(s, label=f'λ={lam}')
plt.axhline(1.0, color='k', lw=0.5, ls='--'); plt.title('s/s* стресс-категории при разных λ')
plt.xlabel('день'); plt.legend(); plt.tight_layout(); plt.show()